In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Define the model ID for MedGemma-27B
model_id = "google/medgemma-27b-text-it"

# Define the quantization configuration for 4-bit loading
# This configuration helps to optimize memory usage and can speed up computation
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for faster computation
)

# Load the tokenizer for the model
# The tokenizer prepares your text input for the model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model with the specified quantization configuration
# `device_map="auto"` automatically places the model on the available GPU(s)
print("Loading model... This may take a few minutes.")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="cuda",
)

print("Model loaded successfully!")

# --- Ready to Generate Text ---

# Create a sample prompt. The model is instruction-tuned, so framing your
# input as a question or instruction is a good practice.
prompt = "What are the potential benefits and risks of using GLP-1 receptor agonists for weight management in patients without diabetes?"

# Format the input using the model's chat template
input_text = tokenizer.apply_chat_template([
    {"role": "user", "content": prompt}
], tokenize=True, add_generation_prompt=True, return_tensors="pt")

In [ ]:
input_text = torch.stack([text.to(model.device) for text in input_text])

In [ ]:
# Tokenize the input text
# input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate a response from the model
# You can adjust parameters like `max_new_tokens` to control the output length
print("\nGenerating response...")
outputs = model.generate(
    input_text,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

# Decode and print the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the clean output
print("\n--- Model Response ---")
# The output will include your original prompt, so we can clean it up for display
response_only = generated_text.split("<start_of_turn>model")[-1].strip()
print(response_only)
print("--------------------\n")